<a href="https://colab.research.google.com/github/Akshaya200722/deep-learning/blob/main/RNN_Forward_Pass_and_BackwardPropagation_with_random_weights.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(suppress=True)

#--------------------
# Define softmax
#--------------------
def softmax(x):
    exp_x = np.exp(x - np.max(x))
    return exp_x / np.sum(exp_x)

#--------------------
# Cross-entropy loss
#--------------------
def cross_entropy(y_true, y_pred):
    return -np.sum(y_true * np.log(y_pred + 1e-12))

#--------------------
# Initialize RNN parameters
#--------------------
Wxh = np.array([[0.1, 0.2, 0.3],
                [0.4, 0.5, 0.6]])

Whh = np.array([[0.7, 0.8],
                [0.9, 1.0]])

Why = np.array([[0.1, 0.2],
                [0.3, 0.4],
                [0.5, 0.6]])

bh = np.array([[0.1],
               [0.2]])

by = np.array([[0.1],
               [0.2],
               [0.3]])

#--------------------
# Inputs (abc)
#--------------------
x1 = np.array([[1],[0],[0]])
x2 = np.array([[0],[1],[0]])
x3 = np.array([[0],[0],[1]])

inputs = [x1, x2, x3]

# Targets (predict next character)
t1 = np.array([[0],[1],[0]])
t2 = np.array([[0],[0],[1]])
t3 = np.array([[1],[0],[0]])

targets = [t1, t2, t3]

#--------------------
# Training hyperparameters
#--------------------
learning_rate = 0.1
threshold = 0.05
max_epochs = 1000
clip_value = 5

def clip_gradients(grad, clip_value):
    return np.clip(grad, -clip_value, clip_value)

loss_history = []

# =============================
# TRAINING LOOP
# =============================
for epoch in range(max_epochs):

    #---- Forward Pass---
    h0 = np.zeros((2,1))
    hs = [h0]
    ps = []
    loss = 0

    for t in range(3):
        ht = np.tanh(Wxh @ inputs[t] + Whh @ hs[t] + bh)
        y = Why @ ht + by
        p = softmax(y)

        hs.append(ht)
        ps.append(p)

        loss += cross_entropy(targets[t], p)

    loss_history.append(loss)

    if loss < threshold:
        print("Training converged at epoch:", epoch+1)
        break

    #---- Backpropagation Through Time---
    dWxh = np.zeros_like(Wxh)
    dWhh = np.zeros_like(Whh)
    dWhy = np.zeros_like(Why)
    dbh = np.zeros_like(bh)
    dby = np.zeros_like(by)

    dh_next = np.zeros((2,1))

    for t in reversed(range(3)):
        dy = ps[t] - targets[t]

        dWhy += dy @ hs[t+1].T
        dby += dy

        dh = Why.T @ dy + dh_next

        dh_raw = (1 - hs[t+1]**2) * dh

        dbh += dh_raw
        dWxh += dh_raw @ inputs[t].T
        dWhh += dh_raw @ hs[t].T

        dh_next = Whh.T @ dh_raw

    #---- Gradient Clipping---
    dWxh = clip_gradients(dWxh, clip_value)
    dWhh = clip_gradients(dWhh, clip_value)
    dWhy = clip_gradients(dWhy, clip_value)
    dbh = clip_gradients(dbh, clip_value)
    dby = clip_gradients(dby, clip_value)

    #---- Update Parameters---
    Wxh -= learning_rate * dWxh
    Whh -= learning_rate * dWhh
    Why -= learning_rate * dWhy
    bh -= learning_rate * dbh
    by -= learning_rate * dby

    if (epoch+1) % 100 == 0:
        print("Epoch:", epoch+1, "Loss:", loss)

print("\nFinal Loss:", loss)

#--------------------
# Final Predictions
#--------------------
print("\nFinal Predictions:")
for t in range(3):
    print(np.round(ps[t], 1))

#--------------------
# Plot Loss Curve
#--------------------
plt.plot(loss_history, color='blue')
plt.title("Training Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()